# K1足球坐标静态只读测试

目的：Ball框 → 深度与内参 → 相机三维球心 → get_transform → trunk候选坐标。

保持机器人、头部和球静止。本程序只查询数据，不发送头部、身体、模式或踢球命令。无需教师JSON。

默认深度比例0.001只是毫米候选假设；同像素配准和深度表示光轴Z也需现场核对。计算成功仍是候选值。


In [ ]:
from pathlib import Path
from probe_ball_coordinates import run_probe


## 填写本次参数

先把一颗球放在视野内。`KNOWN_XY=None`可先观察坐标；知道trunk原点和轴向后再填写卷尺值。

现有detector应为soccer模型；如果不是，将下一格的DETECTOR设为None，由工具创建足球检测器。


In [ ]:
DEPTH_SCALE_M = 0.001    # 待核对的米/原始深度单位，不能靠dtype确认
KNOWN_XY = None           # 可选：(实测x, 实测y)，单位米，使用本次目标帧原点
BALL_RADIUS_M = None      # 可选：实测足球半径，单位米
NOTE = "正前方静止球"    # 每次改为本次位置和观察情况
OUTPUT_ROOT = "足球坐标测试结果"

ROBOT = globals().get("robot")
DETECTOR = globals().get("detector")


## 采集一次并计算

复用已有robot/detector；未传入时由模块创建并缓存。先取RGB、深度和TF，后执行检测。

结束时打印记录目录与ZIP，错误时也保存已有数据。本次检测可能耗时约数秒。


In [ ]:
report = run_probe(
    robot=ROBOT, detector=DETECTOR,
    depth_scale_m=DEPTH_SCALE_M, known_xy=KNOWN_XY,
    ball_radius_m=BALL_RADIUS_M, note=NOTE, output_root=OUTPUT_ROOT,
)


## 核对选中的球与数值

`candidate_only`表示候选值已经算出，不代表已可供踢球控制。

将球移到左侧、右侧或向前移动卷尺量得的距离，停稳后修改NOTE并重运行参数格和采集格。比较左右符号及前后差值；球位变化时，机器人与头部保持不动。


In [ ]:
from IPython.display import display, Image as DisplayImage

folder = Path(report["output_dir"])
picture = folder / "selected_ball.png"
if not picture.exists():
    picture = folder / "rgb.png"
if picture.exists():
    display(DisplayImage(filename=str(picture)))

display({
    "状态": report["status"],
    "候选球心[x,y,z]米": report["candidate_center_target_m"],
    "拟合诊断值（可能已拒用）": report.get("diagnostic_center_target_m"),
    "拒用原因": report.get("sphere_fit_rejection_reasons"),
    "球面拟合": report.get("sphere_fit"),
    "卷尺比较": report.get("measurement_comparison"),
    "RGB/深度时间差秒": report.get("rgb_depth_skew_s"),
    "TF/RGB时间差秒": report.get("transform", {}).get("tf_minus_rgb_s"),
    "检测耗时秒": report.get("detection_elapsed_s"),
    "提示": report["warnings"],
    "异常": report.get("error"),
})


若需要回读分析，保留并反馈本次ZIP即可。它包含原图、原单位深度、内参、矩阵、计算结果和异常。

trunk候选坐标与SoccerKickManager当前控制身体坐标的关系仍需确认。当前静态测试不会将结果传入update_ball，也不承担运动中的时延补偿。
